# 09 · RNN_aug — 내부 시계열 증강 + HR-loss RNN (EXP #36, LB 0.6920)

직전 base = K15/GRU Soft(LB 0.6916). 외부 LB 0.6926 솔루션의 묶음(L24 내부증강 + L25 per-step Frenet + L22 soft-hit loss)을 이식한다.

**가정**: 한 trajectory 내부의 sliding-window로 **진짜 새 (X,y) pair**를 만들고(좌표 불변 증강과 달리 단순 upweighting이 아님), per-step Frenet 9채널 + HR-surrogate loss를 GRU에 주면, K15/GRU와 다른 신호를 만들어 selector로 추가 회수한다.

**실험**: 신규 RNN_aug를 K15·GRU에 얹어 3-way multiclass selector. leakage-free fold(증강 윈도우는 자기 traj fold 상속). 증강으로 OOF 크기가 달라 **판정은 LB-only**. Phase 1 abort gate(solo OOF ≥ GRU 0.6659)로 조기 차단.

**결과**: RNN_aug solo OOF 0.6742(minority 최강)이나 잔차 corr 0.983으로 K15/GRU와 near-redundant, selector acc ≈ chance → **rnn_solo 채택 LB 0.6920 (+0.0004, sub-std)**. plateau(~0.692) 재확인 — correlated 모델 추가로는 못 넘고 **OOF→LB 전이되는 저상관 새 물리**가 필요하다는 결론.

## 셀 0 — imports + 상수 + GPU

In [ ]:
import os, time, json, math, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01
SEEDS = [42, 7, 123]

# L24 augmentation
OUTLIER_V = 1.4          # 윈도우 내 v_scalar 이 이상이면 학습 pool 에서 drop
EPS = 1e-8

# RNN (외부 0.6926 = plain last-hidden, 2 layer)
BACKBONE = 'gru'         # 'gru' | 'lstm'
INPUT_DIM = 9
HIDDEN = 64
NUM_LAYERS = 2
OUT_DIM = 3

# L22 MarginSoftRHit + 학습 (외부 default)
ALPHA = 0.35
K_SOFT = 350.0
LR = 2e-3
BATCH = 128
EPOCHS = 50
PATIENCE = 15
GRAD_CLIP = 1.0

# Gate (RESEARCH §8)
GRU_REF_OVERALL = 0.6659   # #25 V2 Frenet GRU solo OOF (3-seed) — Phase 1 abort gate
GRU_REF_MINOR   = 0.3181
K15_REF_HR = dict(overall=0.6649, major=0.7312, minor=0.3105)   # #31 셀 5.5
ORACLE_KG_GAIN = 0.0221    # #32 K15/GRU 2-way oracle Δo (G_RNN3 비교)
FLOOR_HR_BASE = 0.6688     # #35 K15/GRU soft T=0.5 OOF (현 base, LB 0.6916) — Phase 3 Floor
LB_BASE = 0.6916           # 현 best LB (#35 T=0.5) — 신규 채택은 LB > 0.6916
COMBINED_STD = 0.0014
G4_THR = COMBINED_STD * (2 ** 0.5)   # ≈ 0.0020 per-mode LB 자격

# classifier (LGBM multiclass, #32 binary 와 동일 hyperparam family)
clf_params = dict(
    objective='multiclass', num_class=3, metric='multi_logloss',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=10,
    max_bin=511, n_estimators=300, random_state=42, verbose=-1,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device = {DEVICE}, backbone = {BACKBONE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  torch {torch.__version__}')
os.makedirs('results', exist_ok=True)
print('imports + 상수 OK')

## 셀 1 — helper (Frenet/kinematics + hit_rate + splits + seed)

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def hr_triple(hit_arr, minority_mask):
    return dict(
        overall=float(hit_arr.mean()),
        major=float(hit_arr[~minority_mask].mean()),
        minor=float(hit_arr[minority_mask].mean()),
    )


def make_splits(stratify_int, seed, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(stratify_int)), stratify_int)]


def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]; a_last = a[:, -1, :]
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), a_sm.astype(np.float32)


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    except Exception:
        pass


print('helper 정의 완료')

## 셀 2 — Drive mount + 데이터 + K15(e29)/GRU(e25) cache 동기화

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'
DRIVE_BASE = '/content/drive/MyDrive'

K15_OOF_PATHS  = [f'results/topk_e29_oof_K15_seed{s}.npy' for s in SEEDS]
K15_TEST_PATHS = [f'results/topk_e29_test_K15_seed{s}.npy' for s in SEEDS]
GRU_OOF_PATHS  = [f'results/gru_e25_fren_oof_seed{s}.npy' for s in SEEDS]
GRU_TEST_PATHS = [f'results/gru_e25_fren_test_seed{s}.npy' for s in SEEDS]
all_required = K15_OOF_PATHS + K15_TEST_PATHS + GRU_OOF_PATHS + GRU_TEST_PATHS

need_drive = not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS))
need_cache = any(not os.path.exists(p) for p in all_required)
if need_drive or need_cache:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
if need_drive:
    if not os.path.exists('/content/open'):
        !unzip -q -o "{DRIVE_BASE}/open.zip" -d /content/

for p in all_required:
    if not os.path.exists(p):
        src = f'{DRIVE_BASE}/{p}'
        assert os.path.exists(src), f'cache 누락 (필수): {src}'
        shutil.copy(src, p)
missing = [p for p in all_required if not os.path.exists(p)]
assert not missing, f'K15/GRU cache 누락: {missing}'

def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv', f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')

DATA_DIR = _resolve_data_dir()
if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR); y_train = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        traj_train[i] = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train); np.save(CACHE_Y_TR, y_train)

sample_sub = pd.read_csv(_resolve_sample_sub(DATA_DIR))
test_ids = sample_sub['id'].tolist()
if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        traj_test[i] = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}, cache 12/12 OK')

## 셀 3 — base + Frenet frame + minority mask + V3 15 + K15/GRU 예측 로드

K15/GRU cache 는 residual (Cart/Frenet). base + residual → 절대 예측 → err. selector·oracle 입력.

In [ ]:
FEAT_NAMES_V3_15 = ['a_tang_last','va_dot','speed_last','j_L','cos_turn_last',
    'j_N','a_cent_last','speed_diff_half','turn_mean_half_diff','acc_norm_last',
    'radial_v','turn_mean','acc_norm_w3','distance_r','vy_std']

def build_v3_15(traj, eL, eN, eB):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w_sm = np.array([1, 2, 3]) / 6
    a_w3 = (a[:, -3:, :] * w_sm[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    a_tang_last = (v_last * a_last).sum(1) / (speed_last + 1e-9)
    a_cent_last = _norm(np.cross(v_last, a_last)) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    vy_std = v[:, :, 1].std(axis=1)
    j_L = (jerk_last * eL).sum(1); j_N = (jerk_last * eN).sum(1)
    return np.column_stack([a_tang_last, va_dot, speed_last, j_L, cos_turn_last,
        j_N, a_cent_last, speed_diff_half, turn_mean_half_diff, acc_norm_last,
        radial_v, turn_mean, acc_norm_w3, distance_r, vy_std]).astype(np.float32)

base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
vl_tr, al_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)
minority_mask_tr = _norm(al_tr) >= MINORITY_THRESHOLD
minority_mask_ts = _norm(al_ts) >= MINORITY_THRESHOLD
print(f'base train HR (sanity 0.5788): {hit_rate(base_train, y_train):.4f}')
print(f'minority train {minority_mask_tr.sum()}/{len(minority_mask_tr)}')

X_tr_v15 = build_v3_15(traj_train, eL_tr, eN_tr, eB_tr)
X_ts_v15 = build_v3_15(traj_test,  eL_ts, eN_ts, eB_ts)

def load_seed_avg(paths):
    return np.mean([np.load(p) for p in paths], axis=0).astype(np.float32)

# K15 (Cart residual)
oof_K15 = load_seed_avg(K15_OOF_PATHS); test_K15 = load_seed_avg(K15_TEST_PATHS)
pred_K15_tr = base_train + oof_K15
pred_K15_ts = base_test + test_K15
err_K15 = np.linalg.norm(y_train - pred_K15_tr, axis=1)
HR_K15 = hr_triple(err_K15 < HIT_RADIUS, minority_mask_tr)
print(f'K15 HR: {HR_K15}')
for k in ['overall','major','minor']:
    assert abs(HR_K15[k] - K15_REF_HR[k]) < 0.0008, f'K15 {k} drift {HR_K15[k]} vs {K15_REF_HR[k]}'

# GRU (Frenet residual → Cart inverse; #32 auto-detect 와 동일)
oof_GRU = load_seed_avg(GRU_OOF_PATHS); test_GRU = load_seed_avg(GRU_TEST_PATHS)
oof_GRU_cart = (oof_GRU[:,0:1]*eL_tr + oof_GRU[:,1:2]*eN_tr + oof_GRU[:,2:3]*eB_tr)
test_GRU_cart = (test_GRU[:,0:1]*eL_ts + test_GRU[:,1:2]*eN_ts + test_GRU[:,2:3]*eB_ts)
pred_GRU_tr = base_train + oof_GRU_cart
pred_GRU_ts = base_test + test_GRU_cart
err_GRU = np.linalg.norm(y_train - pred_GRU_tr, axis=1)
HR_GRU = hr_triple(err_GRU < HIT_RADIUS, minority_mask_tr)
print(f'GRU HR (sanity overall≈{GRU_REF_OVERALL}): {HR_GRU}')
assert abs(HR_GRU['overall'] - GRU_REF_OVERALL) < 0.005, 'GRU cache 형태 불일치 (Frenet 가정 fail)'

## 셀 4 — L24 증강 + 6점 per-step Frenet extractor (신규)

`frenet_seq(win)`: win (M,6,3) → step i=2..5 per-step Frenet 9채널 시퀀스 (M,4,9) + R_last(M,3,3) + v0(M,) + vmax(M,).
`build_windows`: real 윈도우(idx5:11) + (옵션) 내부 4 윈도우 → 통합 array + traj_id + is_real + residual target.
외부 코드(셀 4) 비트 매칭: cross_norm≤eps 시 B=N=0 (frame 부분 singular 허용).

In [ ]:
def frenet_seq(win):
    """win (M,6,3) → feat (M,4,9), R_last (M,3,3), v0 (M,), vmax (M,)."""
    M = win.shape[0]
    feats = []
    R_prev = None; T_prev = None; R_last = None; v0 = None
    vmax = np.zeros(M, dtype=np.float64)
    for i in range(2, 6):
        p0, pm1, pm2 = win[:, i], win[:, i-1], win[:, i-2]
        v = (p0 - pm1) / DT
        a = (p0 - 2*pm1 + pm2) / (DT*DT)
        vn = np.linalg.norm(v, axis=1) + EPS
        an = np.linalg.norm(a, axis=1) + EPS
        cross = np.cross(v, a); cn = np.linalg.norm(cross, axis=1) + EPS
        vda = (v * a).sum(1)
        T = v / vn[:, None]
        kappa = np.where(cn > EPS, cn / np.maximum(vn**3, EPS), 0.0)
        nondeg = cn > EPS
        B = np.where(nondeg[:, None], cross / cn[:, None], 0.0)
        N = np.where(nondeg[:, None], np.cross(B, T), 0.0)
        R_curr = np.stack([T, N, B], axis=2)   # columns = [T,N,B]
        cos_phi = np.clip(vda / (vn * an), -1.0, 1.0); phi = np.arccos(cos_phi)
        log_kappa = np.log(kappa + EPS)
        log_ratio = np.where(vda > 0,
                             np.log(np.maximum(kappa * (vn**3) / (vda + EPS), EPS)),
                             np.log(EPS))
        if R_prev is None:
            theta = np.zeros(M); dT = np.zeros((M, 3))
        else:
            R_rel = np.matmul(np.transpose(R_prev, (0, 2, 1)), R_curr)
            tr = np.trace(R_rel, axis1=1, axis2=2)
            theta = np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0))
            dT = T - T_prev
        feats.append(np.column_stack([vn - EPS, vda, log_kappa, log_ratio, theta, phi, dT]))
        vmax = np.maximum(vmax, vn - EPS)
        R_prev = R_curr; T_prev = T; R_last = R_curr; v0 = vn - EPS
    feat = np.stack(feats, axis=1).astype(np.float32)   # (M,4,9)
    return feat, R_last.astype(np.float32), v0.astype(np.float32), vmax.astype(np.float32)


def build_windows(traj_arr, y_arr, include_internal):
    """return dict of (M,...) arrays. real window 항상 포함; include_internal 시 내부 4 추가."""
    N = len(traj_arr)
    blocks = [(traj_arr[:, 5:11, :], y_arr, True)]   # real: idx5..10 → 외부 라벨
    if include_internal:
        for w in range(4):
            blocks.append((traj_arr[:, w:w+6, :], traj_arr[:, w+7, :], False))
    feat_l, R_l, p0_l, v0_l, tgt_l, tid_l, real_l, vmax_l = [], [], [], [], [], [], [], []
    for win, tgt, is_real in blocks:
        f, R, v0, vmax = frenet_seq(win)
        feat_l.append(f); R_l.append(R); v0_l.append(v0); vmax_l.append(vmax)
        p0_l.append(win[:, 5, :].astype(np.float32))
        tid_l.append(np.arange(N))
        real_l.append(np.full(N, is_real, dtype=bool))
        tgt_l.append(tgt.astype(np.float32) if tgt is not None else np.full((N, 3), np.nan, np.float32))
    feat = np.concatenate(feat_l); R = np.concatenate(R_l); p0 = np.concatenate(p0_l)
    v0 = np.concatenate(v0_l); tgt = np.concatenate(tgt_l); tid = np.concatenate(tid_l)
    is_real = np.concatenate(real_l); vmax = np.concatenate(vmax_l)
    # residual target (local): R^T @ (tgt - p0) - [v0*DT_PRED, 0, 0]
    delta = tgt - p0
    tl = np.einsum('mik,mi->mk', R, delta)   # R^T @ delta (R columns = axes)
    bl = np.zeros((len(R), 3), dtype=np.float32); bl[:, 0] = v0 * DT_PRED
    res = (tl - bl).astype(np.float32)
    return dict(feat=feat, R=R, p0=p0.astype(np.float32), v0=v0.astype(np.float32),
                tgt=tgt, res=res, tid=tid, is_real=is_real, vmax=vmax)


W_tr = build_windows(traj_train, y_train, include_internal=True)
W_ts = build_windows(traj_test, None, include_internal=False)
print(f'train windows: {len(W_tr["feat"])} (real {int(W_tr["is_real"].sum())} + 내부 {int((~W_tr["is_real"]).sum())})')
print(f'test windows: {len(W_ts["feat"])} (real only)')
drop = W_tr['vmax'] >= OUTLIER_V
print(f'outlier(v≥{OUTLIER_V}) train 윈도우: {int(drop.sum())} ({drop.mean()*100:.1f}%) — 학습 pool 에서 제외')

# Sanity: real window 만 골라 base HR + residual reconstruction
real = W_tr['is_real']
assert real.sum() == 10000
p0r, Rr, v0r, resr = W_tr['p0'][real], W_tr['R'][real], W_tr['v0'][real], W_tr['res'][real]
bl = np.zeros((10000, 3), np.float32); bl[:, 0] = v0r * DT_PRED
recon = p0r + np.einsum('mik,mk->mi', Rr, bl + resr)
ord_tid = np.argsort(W_tr['tid'][real])   # traj_id 순 정렬 (=0..9999)
assert np.array_equal(W_tr['tid'][real][ord_tid], np.arange(10000))
rec_err = np.linalg.norm(recon - y_train[W_tr['tid'][real]], axis=1)
print(f'real-window residual reconstruction err (≈0 except singular frame): median {np.median(rec_err)*1000:.4f}mm, p99 {np.percentile(rec_err,99)*1000:.3f}mm')
# 6점 per-step baseline HR (real window) — 11점 base 와 다른 frame 이라 정확히 0.5788 아님
base6 = p0r + np.einsum('mik,mk->mi', Rr, bl)
print(f'6점 per-step CV baseline HR (real): {hit_rate(base6, y_train[W_tr["tid"][real]]):.4f}')

## 셀 5 — RNN model (GRU/LSTM) + MarginSoftRHit loss + train_one_fold

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, backbone=BACKBONE, input_dim=INPUT_DIM, hidden=HIDDEN,
                 num_layers=NUM_LAYERS, out_dim=OUT_DIM):
        super().__init__()
        rnn_cls = nn.GRU if backbone == 'gru' else nn.LSTM
        self.rnn = rnn_cls(input_dim, hidden, num_layers=num_layers, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, out_dim))
    def forward(self, seq):
        out, _ = self.rnn(seq)
        return self.head(out[:, -1, :])


def reconstruct_global(pred_res, p0, v0, R):
    bl = torch.zeros_like(pred_res); bl[:, 0] = v0 * DT_PRED
    total = bl + pred_res
    disp = torch.matmul(R, total.unsqueeze(-1)).squeeze(-1)
    return p0 + disp


def margin_soft_rhit(pred_res, tgt_res, p0, v0, R, tgt_glob, alpha=ALPHA, k=K_SOFT):
    l1 = (pred_res - tgt_res).abs().mean()
    pg = reconstruct_global(pred_res, p0, v0, R)
    err = torch.sqrt(((pg - tgt_glob) ** 2).sum(-1) + 1e-12)
    soft = torch.sigmoid(k * (err - HIT_RADIUS)).mean()
    return (1.0 - alpha) * l1 + alpha * soft


def _to_t(a):
    return torch.from_numpy(np.ascontiguousarray(a)).float().to(DEVICE)


def train_one_fold(tr, va_real, ts_real, model_seed):
    """tr/va_real/ts_real: dict subsets. val/test 는 real window. returns (oof_global, test_global, best_rhit, n_ep)."""
    scaler = StandardScaler().fit(tr['feat'].reshape(-1, INPUT_DIM))
    def s(x): return scaler.transform(x.reshape(-1, INPUT_DIM)).reshape(x.shape).astype(np.float32)
    S_tr = _to_t(s(tr['feat'])); Rr_tr = _to_t(tr['res']); P_tr = _to_t(tr['p0'])
    V_tr = _to_t(tr['v0']); M_tr = _to_t(tr['R']); G_tr = _to_t(tr['tgt'])
    S_va = _to_t(s(va_real['feat'])); P_va = _to_t(va_real['p0']); V_va = _to_t(va_real['v0']); M_va = _to_t(va_real['R'])
    g_va = va_real['tgt']
    S_ts = _to_t(s(ts_real['feat'])); P_ts = _to_t(ts_real['p0']); V_ts = _to_t(ts_real['v0']); M_ts = _to_t(ts_real['R'])

    set_seed(model_seed)
    model = RNNModel().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    n = len(S_tr)
    best_rhit = -1.0; best_state = None; pc = 0; last_ep = 0
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i+BATCH]
            opt.zero_grad()
            pred = model(S_tr[idx])
            loss = margin_soft_rhit(pred, Rr_tr[idx], P_tr[idx], V_tr[idx], M_tr[idx], G_tr[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        model.eval()
        with torch.no_grad():
            pg = reconstruct_global(model(S_va), P_va, V_va, M_va).cpu().numpy()
        rhit = float((np.linalg.norm(pg - g_va, axis=1) < HIT_RADIUS).mean())
        if rhit > best_rhit:
            best_rhit = rhit
            best_state = {kk: vv.detach().clone() for kk, vv in model.state_dict().items()}
            pc = 0
        else:
            pc += 1
            if pc >= PATIENCE:
                break
        last_ep = ep
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        oof_g = reconstruct_global(model(S_va), P_va, V_va, M_va).cpu().numpy().astype(np.float32)
        test_g = reconstruct_global(model(S_ts), P_ts, V_ts, M_ts).cpu().numpy().astype(np.float32)
    return oof_g, test_g, best_rhit, last_ep + 1


_m = RNNModel(); print(f'RNNModel({BACKBONE}) params: {sum(p.numel() for p in _m.parameters()):,}'); del _m

## 셀 6 — Phase 1: RNN_aug 학습 (trajectory-level fold) + abort gate G_RNN1

seed 42 먼저 학습 → **G_RNN1: solo OOF ≥ 0.6659** 미달 시 STOP. 통과 시 seed 7,123.
fold = trajectory index StratifiedKFold(stratify=minority) = K15/GRU 동일. 학습 pool = train traj 의 모든 윈도우 중 vmax<1.4. OOF = val traj 의 real window.

In [ ]:
def subset_W(W, mask):
    return {kk: W[kk][mask] for kk in ['feat','R','p0','v0','tgt','res','tid','is_real','vmax']}

ts_real = subset_W(W_ts, np.ones(len(W_ts['feat']), bool))   # 전부 real
tid_all = W_tr['tid']; real_all = W_tr['is_real']; vmax_all = W_tr['vmax']

def run_rnn_seed(seed):
    folds = make_splits(minority_mask_tr.astype(int), seed=seed)   # traj index 분할
    oof_g = np.full((10000, 3), np.nan, dtype=np.float32)
    test_g = np.zeros((10000, 3), dtype=np.float32)
    rhits, eps_used = [], []
    t_seed = time.time()
    for fi, (tr_traj, va_traj) in enumerate(folds):
        tr_set = set(tr_traj.tolist()); va_set = set(va_traj.tolist())
        tr_mask = np.array([t in tr_set for t in tid_all]) & (vmax_all < OUTLIER_V)
        va_mask = np.array([t in va_set for t in tid_all]) & real_all
        tr = subset_W(W_tr, tr_mask); va = subset_W(W_tr, va_mask)
        oof_f, test_f, rhit, n_ep = train_one_fold(tr, va, ts_real, model_seed=seed*1000+fi)
        oof_g[va['tid']] = oof_f
        test_g += test_f / N_FOLDS
        rhits.append(rhit); eps_used.append(n_ep)
        print(f'    seed={seed} fold={fi}: ntrain={tr_mask.sum()} val_rhit={rhit:.4f} ep={n_ep} ({time.time()-t_seed:.0f}s cum)')
    HR = hr_triple(np.linalg.norm(oof_g - y_train, axis=1) < HIT_RADIUS, minority_mask_tr)
    np.save(f'results/rnn_aug_e36_oof_seed{seed}.npy', oof_g)
    np.save(f'results/rnn_aug_e36_test_seed{seed}.npy', test_g)
    print(f'  seed={seed} DONE: OOF HR={HR["overall"]:.4f} major={HR["major"]:.4f} minor={HR["minor"]:.4f} mean_ep={np.mean(eps_used):.0f}')
    return oof_g, test_g, HR

print('=== Phase 1 seed 42 (abort gate 선검증) ===')
oof_42, test_42, HR_42 = run_rnn_seed(42)
abort = HR_42['overall'] < GRU_REF_OVERALL
print(f'\nG_RNN1 (solo OOF ≥ {GRU_REF_OVERALL}): seed42 {HR_42["overall"]:.4f} → {"FAIL ✗ STOP" if abort else "PASS ✓"}')
rnn_oof_per_seed = {42: oof_42}; rnn_test_per_seed = {42: test_42}; rnn_HR_per_seed = {42: HR_42}
if abort:
    print('⚠ 증강+HR-loss 가 #25 GRU 강화 실패. Phase 2~4 skip. (lever 재설계 필요)')
else:
    for s in [7, 123]:
        o, t, h = run_rnn_seed(s)
        rnn_oof_per_seed[s] = o; rnn_test_per_seed[s] = t; rnn_HR_per_seed[s] = h

## 셀 7 — Phase 2: 3-way oracle ceiling (K15/GRU/RNN) + pairwise corr

RNN_aug 3-seed mean → pred_global. G_RNN2 corr<0.95, G_RNN3 3-way oracle Δo > 0.0221.

In [ ]:
if not abort:
    rnn_oof = np.mean([rnn_oof_per_seed[s] for s in SEEDS], axis=0).astype(np.float32)
    rnn_test = np.mean([rnn_test_per_seed[s] for s in SEEDS], axis=0).astype(np.float32)
    err_RNN = np.linalg.norm(y_train - rnn_oof, axis=1)
    HR_RNN = hr_triple(err_RNN < HIT_RADIUS, minority_mask_tr)
    print(f'RNN_aug 3-seed OOF HR: {HR_RNN}')

    # G_RNN2: pairwise corr (residual 공간)
    rK = (y_train - pred_K15_tr).ravel(); rG = (y_train - pred_GRU_tr).ravel(); rR = (y_train - rnn_oof).ravel()
    corr_KR = float(np.corrcoef(rK, rR)[0,1]); corr_GR = float(np.corrcoef(rG, rR)[0,1]); corr_KG = float(np.corrcoef(rK, rG)[0,1])
    print(f'corr K15·RNN={corr_KR:.4f}  GRU·RNN={corr_GR:.4f}  K15·GRU={corr_KG:.4f}')
    if corr_GR >= 0.95:
        print(f'  ⚠ GRU·RNN corr ≥ 0.95 → 3-way 가 2-way(K15/RNN) 대비 GRU 추가 가치 미미 가능')

    # oracle (per-sample best)
    errs = np.stack([err_K15, err_GRU, err_RNN], axis=1)
    preds3 = np.stack([pred_K15_tr, pred_GRU_tr, rnn_oof], axis=1)   # (10000,3,3)
    oracle_arg = errs.argmin(1)
    pred_oracle3 = preds3[np.arange(10000), oracle_arg]
    HR_oracle3 = hr_triple(np.linalg.norm(y_train-pred_oracle3,axis=1) < HIT_RADIUS, minority_mask_tr)
    # 2-way K15/RNN oracle (floor 비교)
    pred_oracle2 = np.where((err_RNN < err_K15)[:,None], rnn_oof, pred_K15_tr)
    HR_oracle2 = hr_triple(np.linalg.norm(y_train-pred_oracle2,axis=1) < HIT_RADIUS, minority_mask_tr)
    d3 = HR_oracle3['overall'] - HR_K15['overall']; d2 = HR_oracle2['overall'] - HR_K15['overall']
    print(f'\n3-way oracle HR={HR_oracle3["overall"]:.4f} (Δo +{d3:.4f})  |  2-way K15/RNN oracle Δo +{d2:.4f}')
    print(f'G_RNN3 (3-way oracle Δo > {ORACLE_KG_GAIN}): {"PASS ✓" if d3 > ORACLE_KG_GAIN else "미달 (prior 하향, Phase 3 진행 가능)"}')
    print(f'3-way 가 2-way 대비 GRU 추가 가치: +{d3-d2:.4f}')

## 셀 8 — Phase 3: multiclass selector (V3 15) + blend (Soft/Hard) + T soften

target = argmin(err_K15, err_GRU, err_RNN) ∈ {0,1,2}. 3 seed × 5 fold (stratify=minority).
Soft = Σ p_i·pred_i, Hard = argmax. T soften: p'=softmax(log p / T). 2-way(K15/RNN) floor 동시.

In [ ]:
blend_modes = {}
if not abort:
    y_sel3 = errs.argmin(1).astype(int)   # 0=K15,1=GRU,2=RNN
    print(f'selector target 분포: K15={int((y_sel3==0).sum())} GRU={int((y_sel3==1).sum())} RNN={int((y_sel3==2).sum())}')
    clf_oof = np.full((10000, 3), np.nan, dtype=np.float32)
    clf_test = np.zeros((10000, 3), dtype=np.float32)
    for si, seed in enumerate(SEEDS):
        folds = make_splits(minority_mask_tr.astype(int), seed=seed)
        oof_s = np.full((10000, 3), np.nan, dtype=np.float32)
        test_s = np.zeros((10000, 3), dtype=np.float32)
        for tr_idx, va_idx in folds:
            m = lgb.LGBMClassifier(**{**clf_params, 'random_state': seed})
            m.fit(X_tr_v15[tr_idx], y_sel3[tr_idx])
            oof_s[va_idx] = m.predict_proba(X_tr_v15[va_idx]).astype(np.float32)
            test_s += m.predict_proba(X_ts_v15).astype(np.float32) / N_FOLDS
        np.save(f'results/split_e36_clf_oof_seed{seed}.npy', oof_s)
        np.save(f'results/split_e36_clf_test_seed{seed}.npy', test_s)
        clf_oof = oof_s if si == 0 else clf_oof + oof_s
        clf_test += test_s
    clf_oof /= len(SEEDS); clf_test /= len(SEEDS)
    # selector 직접 accuracy (recovery 의미는 oracle 영역에서)
    sel_acc = float((clf_oof.argmax(1) == y_sel3).mean())
    print(f'selector OOF argmax accuracy = {sel_acc:.4f}')

    preds3_oof = np.stack([pred_K15_tr, pred_GRU_tr, rnn_oof], axis=1)        # (10000,3,3)
    preds3_test = np.stack([pred_K15_ts, pred_GRU_ts, rnn_test], axis=1)
    def soften(p, T):
        lp = np.log(p + 1e-9) / T
        e = np.exp(lp - lp.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
    def blend_soft(prob, preds):
        return (prob[:, :, None] * preds).sum(1)
    def eval_pred(pred):
        return hr_triple(np.linalg.norm(y_train - pred, axis=1) < HIT_RADIUS, minority_mask_tr)

    # Soft (T grid) + Hard + 2-way floor
    for T in [1.0, 1.5, 2.0]:
        pr = soften(clf_oof, T) if T != 1.0 else clf_oof
        blend_modes[f'soft_T{T}'] = dict(HR=eval_pred(blend_soft(pr, preds3_oof)), T=T, kind='soft')
    blend_modes['hard'] = dict(HR=eval_pred(preds3_oof[np.arange(10000), clf_oof.argmax(1)]), kind='hard')
    # 2-way floor: K15/RNN binary soft (p_RNN / (p_K15+p_RNN))
    p2 = clf_oof[:, 2] / (clf_oof[:, 0] + clf_oof[:, 2] + 1e-9)
    pred2 = p2[:, None] * rnn_oof + (1 - p2[:, None]) * pred_K15_tr
    blend_modes['floor2way'] = dict(HR=eval_pred(pred2), kind='2way')

    print(f'\n{"mode":<12}{"overall":>9}{"major":>9}{"minor":>9}{"Δo vs K15":>11}{"Δo vs base":>11}')
    for name, info in blend_modes.items():
        H = info['HR']; dK = H['overall']-HR_K15['overall']; d33 = H['overall']-FLOOR_HR_BASE
        info['d_K15'] = dK; info['d_floor33'] = d33
        print(f'{name:<12}{H["overall"]:>9.4f}{H["major"]:>9.4f}{H["minor"]:>9.4f}{dK:>+11.4f}{d33:>+11.4f}')

## 셀 9 — Gate 판정 + Phase 4 submission (best mode)

Floor: Δo > #33 OOF 0.6687 (현 base) AND > 2-way floor. 4th guard: Δo vs K15 > +0.0020.
G2 major>−0.002, G3 minor>−0.005. 통과 mode 중 best overall 1개만 submission (LB 1슬롯).

In [ ]:
best_mode = None; sub_path = None
if not abort:
    floor2 = blend_modes['floor2way']['HR']['overall']
    cands = []
    for name, info in blend_modes.items():
        if name == 'floor2way':
            continue
        H = info['HR']
        g_floor33 = H['overall'] > FLOOR_HR_BASE
        g_floor2  = H['overall'] > floor2
        g4 = info['d_K15'] > G4_THR
        g2 = (H['major'] - HR_K15['major']) > -0.002
        g3 = (H['minor'] - HR_K15['minor']) > -0.005
        ok = g_floor33 and g_floor2 and g4 and g2 and g3
        info['gates'] = dict(floor33=g_floor33, floor2way=g_floor2, g4=g4, g2=g2, g3=g3, pass_all=ok)
        flag = 'O' if ok else 'X'
        print(f'  {name:<10} Δo(K15)={info["d_K15"]:+.4f} floor33={"O" if g_floor33 else "X"} '
              f'floor2way={"O" if g_floor2 else "X"} g4={"O" if g4 else "X"} g2={"O" if g2 else "X"} g3={"O" if g3 else "X"} → {flag}')
        if ok:
            cands.append((name, H['overall']))
    if cands:
        best_mode = max(cands, key=lambda c: c[1])[0]
        print(f'\n채택 mode: {best_mode} (OOF {blend_modes[best_mode]["HR"]["overall"]:.4f}) → LB 1슬롯')
    else:
        print('\n채택 없음 — 모든 mode Floor/4th guard 미달. LB skip.')

if best_mode is not None:
    info = blend_modes[best_mode]
    def soften_t(p, T):
        lp = np.log(p + 1e-9) / T; e = np.exp(lp - lp.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
    if info.get('kind') == 'soft':
        T = info['T']; pr = soften_t(clf_test, T) if T != 1.0 else clf_test
        pred_test = (pr[:, :, None] * preds3_test).sum(1)
    elif info.get('kind') == 'hard':
        pred_test = preds3_test[np.arange(10000), clf_test.argmax(1)]
    else:
        p2t = clf_test[:, 2] / (clf_test[:, 0] + clf_test[:, 2] + 1e-9)
        pred_test = p2t[:, None] * rnn_test + (1 - p2t[:, None]) * pred_K15_ts
    pred_test = pred_test.astype(np.float32)
    assert pred_test.shape == (10000, 3) and np.isfinite(pred_test).all()
    sub_path = f'submission_rnn_aug_e36_{best_mode}.csv'
    pd.DataFrame({'id': test_ids, 'x': pred_test[:,0], 'y': pred_test[:,1], 'z': pred_test[:,2]}).to_csv(sub_path, index=False)
    print(f'submission: {sub_path}')
    print(f'  x[{pred_test[:,0].min():.2f},{pred_test[:,0].max():.2f}] y[{pred_test[:,1].min():.2f},{pred_test[:,1].max():.2f}] z[{pred_test[:,2].min():.2f},{pred_test[:,2].max():.2f}]')

## 셀 10 — Meta 저장 + Drive 복사 + 로컬 다운로드

In [ ]:
def deep_safe(d):
    if isinstance(d, dict): return {k: deep_safe(v) for k, v in d.items()}
    if isinstance(d, (list, tuple)): return [deep_safe(x) for x in d]
    if isinstance(d, np.floating): return float(d)
    if isinstance(d, np.integer): return int(d)
    if isinstance(d, (np.bool_, bool)): return bool(d)
    if isinstance(d, np.ndarray): return d.tolist()
    return d

meta = {
    'protocol': 'EXP #36 RNN_aug (L24 aug + L25 per-step Frenet + L22 softhit) → 3-way selector',
    'config': dict(backbone=BACKBONE, hidden=HIDDEN, num_layers=NUM_LAYERS,
                   alpha=ALPHA, k_soft=K_SOFT, lr=LR, batch=BATCH, epochs=EPOCHS,
                   patience=PATIENCE, outlier_v=OUTLIER_V),
    'phase1': dict(
        abort=bool(abort),
        rnn_HR_per_seed=deep_safe({s: rnn_HR_per_seed.get(s) for s in rnn_HR_per_seed}),
        gate_grnn1=dict(ref=GRU_REF_OVERALL, seed42=float(HR_42['overall'])),
    ),
}
if not abort:
    meta['phase2'] = deep_safe(dict(rnn_HR=HR_RNN, corr=dict(K15_RNN=corr_KR, GRU_RNN=corr_GR, K15_GRU=corr_KG),
                                    oracle3=HR_oracle3, oracle2=HR_oracle2, d3=d3, d2=d2))
    meta['phase3'] = deep_safe({k: dict(HR=v['HR'], d_K15=v.get('d_K15'), d_floor33=v.get('d_floor33'),
                                        gates=v.get('gates')) for k, v in blend_modes.items()})
    meta['phase3']['selector_acc'] = float(sel_acc)
meta['best_mode'] = best_mode
meta['submission_path'] = sub_path
with open('results/rnn_aug_e36_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('results/rnn_aug_e36_meta.json 저장')

try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/rnn_aug_e36_* {results_drive}/
    !cp -r results/split_e36_clf_* {results_drive}/ 2>/dev/null
    !cp results/rnn_aug_e36_meta.json {results_drive}/
    # submission Drive 복사/다운로드는 셀 11(검증 후 PICK)에서 1개만 수행 (best-only)
    files.download('results/rnn_aug_e36_meta.json')
    print('Drive 복사 + 다운로드 완료')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')

## 셀 11 — RNN_aug solo vs soft_T1.5 동시검증 → 더 나은 후보 1개만 submission

§분석 후속. 두 후보 모두 **동일 OOF real-window** 에서 평가 → 상대비교 valid.
soft_T1.5 OOF(0.6741) ≈ RNN solo(0.6742) 로 overall 동률(noise band ±COMBINED_STD).
selector_acc≈chance(0.377)·corr≈0.983 이므로 selector 가 overall 을 못 올림.

**판정 규칙**: overall 차이 ≤ COMBINED_STD(0.0014) → 동률로 보고 **minority tiebreak**,
밴드 밖이면 overall 우위. `FORCE_PICK` 로 수동 override. best-only: PICK 1개만 download.

In [ ]:
## 셀 11 — solo vs soft_T1.5 동시검증 → PICK 1개 submission
assert not abort, 'Phase 1 abort — 후보 없음'
FORCE_PICK = None   # 'rnn_solo' | 'soft_T1.5' | None(auto)

def _soft(p, T):
    lp = np.log(p + 1e-9) / T
    e = np.exp(lp - lp.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

# 후보 OOF (val real-window 동일 기준 → 상대비교 valid)
HR_solo = hr_triple(np.linalg.norm(rnn_oof - y_train, axis=1) < HIT_RADIUS, minority_mask_tr)
oof_sel = (_soft(clf_oof, 1.5)[:, :, None] * preds3_oof).sum(1)
HR_sel  = hr_triple(np.linalg.norm(oof_sel - y_train, axis=1) < HIT_RADIUS, minority_mask_tr)

cand = {
    'rnn_solo':  dict(HR=HR_solo, test=rnn_test.astype(np.float32)),
    'soft_T1.5': dict(HR=HR_sel,
                      test=(_soft(clf_test, 1.5)[:, :, None] * preds3_test).sum(1).astype(np.float32)),
}
print(f'{"cand":<11}{"overall":>9}{"major":>9}{"minor":>9}')
for nm, c in cand.items():
    H = c['HR']; print(f'{nm:<11}{H["overall"]:>9.4f}{H["major"]:>9.4f}{H["minor"]:>9.4f}')

d_o = cand['rnn_solo']['HR']['overall'] - cand['soft_T1.5']['HR']['overall']
d_m = cand['rnn_solo']['HR']['minor']   - cand['soft_T1.5']['HR']['minor']
print(f'\nDelta overall(solo-sel)={d_o:+.4f} (band +-{COMBINED_STD:.4f})  Delta minor={d_m:+.4f}')

if FORCE_PICK:
    pick, rule = FORCE_PICK, 'FORCE'
elif abs(d_o) <= COMBINED_STD:
    pick = 'rnn_solo' if d_m >= 0 else 'soft_T1.5'
    rule = 'overall 동률 -> minority tiebreak'
else:
    pick = 'rnn_solo' if d_o > 0 else 'soft_T1.5'
    rule = 'overall 우위'
print(f'판정 [{rule}] -> 채택: {pick}  (OOF {cand[pick]["HR"]["overall"]:.4f})')

# 단일 submission (best-only)
pt = cand[pick]['test']
assert pt.shape == (10000, 3) and np.isfinite(pt).all()
final_path = f'submission_rnn_aug_e36_PICK_{pick}.csv'
pd.DataFrame({'id': test_ids, 'x': pt[:, 0], 'y': pt[:, 1], 'z': pt[:, 2]}).to_csv(final_path, index=False)
print(f'submission: {final_path}')
print(f'  x[{pt[:,0].min():.2f},{pt[:,0].max():.2f}] '
      f'y[{pt[:,1].min():.2f},{pt[:,1].max():.2f}] z[{pt[:,2].min():.2f},{pt[:,2].max():.2f}]')

# Drive 복사 + 다운로드 (PICK 1개만)
try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    shutil.copy(final_path, f'{DRIVE_BASE}/{final_path}')
    files.download(final_path)
    print('Drive 복사 + 다운로드 완료')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')


## 셀 12 — sub-1.0 T sweep (T=0.5) : #36 grid 공백 보완 [eval-only]

직전 분석 후속. #36 Phase 3 grid 가 [1.0,1.5,2.0] 만 쓰고 **T=0.5(sharpen) 을 누락**해서,
그 영역을 동일 selector(`clf_oof`)·동일 게이트(셀 9)로 메움. 재학습 없음 — 메모리 재사용.
**eval-only**: PICK/submission 미변경. 0.5 가 현 PICK(rnn_solo)을 뒤집으면 그때 제출 셀 추가.

In [ ]:
## 셀 12 — sub-1.0 T sweep (T=0.5) eval-only
assert not abort, 'Phase 1 abort'
T_LOW = [0.5]   # 요청값. 이웃 sharpen 영역도 보려면 [0.3, 0.5, 0.7] 로 확장 가능.

def _softT(p, T):
    lp = np.log(p + 1e-9) / T
    e = np.exp(lp - lp.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

floor2 = blend_modes['floor2way']['HR']['overall']
ref_sel15 = blend_modes['soft_T1.5']['HR']['overall']   # 기존 best soft
ref_solo  = HR_RNN['overall']                           # 현 PICK (rnn_solo)
b = lambda x: 'O' if x else 'X'

print(f'{"mode":<11}{"overall":>9}{"major":>9}{"minor":>9}{"Δo K15":>9}'
      f'{"f33":>5}{"f2":>4}{"g4":>4}{"g2":>4}{"g3":>4}{"pass":>6}')
low = {}
for T in T_LOW:
    blend = (_softT(clf_oof, T)[:, :, None] * preds3_oof).sum(1)
    H = hr_triple(np.linalg.norm(y_train - blend, axis=1) < HIT_RADIUS, minority_mask_tr)
    dK = H['overall'] - HR_K15['overall']
    g_f33 = H['overall'] > FLOOR_HR_BASE
    g_f2  = H['overall'] > floor2
    g4 = dK > G4_THR
    g2 = (H['major'] - HR_K15['major']) > -0.002
    g3 = (H['minor'] - HR_K15['minor']) > -0.005
    ok = g_f33 and g_f2 and g4 and g2 and g3
    low[T] = dict(HR=H, pass_all=ok)
    print(f'soft_T{T:<5}{H["overall"]:>9.4f}{H["major"]:>9.4f}{H["minor"]:>9.4f}{dK:>+9.4f}'
          f'{b(g_f33):>5}{b(g_f2):>4}{b(g4):>4}{b(g2):>4}{b(g3):>4}{b(ok):>6}')

print(f'\nref  soft_T1.5={ref_sel15:.4f}  rnn_solo(현 PICK)={ref_solo:.4f}  (minor solo={HR_RNN["minor"]:.4f})')
bT = max(low, key=lambda t: low[t]['HR']['overall'])
bH = low[bT]['HR']
d_solo = bH['overall'] - ref_solo
if abs(d_solo) <= COMBINED_STD:
    flip = (bH['minor'] >= HR_RNN['minor'])   # 동률 → minority tiebreak (셀 11 규칙)
    verdict = f'soft_T{bT} 재검토(minority 우위)' if flip else 'rnn_solo 유지'
else:
    flip = d_solo > 0
    verdict = f'soft_T{bT} 우위' if flip else 'rnn_solo 유지'
print(f'sub-1.0 best soft_T{bT}={bH["overall"]:.4f} minor={bH["minor"]:.4f} pass={low[bT]["pass_all"]}')
print(f'Δoverall(sub1.0best − solo)={d_solo:+.4f} (band ±{COMBINED_STD:.4f}) → {verdict}')
print('[eval-only] submission 미변경. flip 시 셀 11 FORCE_PICK 또는 별도 제출 셀로 진행.')
